In [ ]:
import ast

import pandas as pd

from scipy.stats import fisher_exact, chi2_contingency
from statsmodels.stats.multitest import multipletests


In [ ]:
df = pd.read_csv("whisper_medium_predictions.csv")

def normalise_language(lang):
    lang = str(lang).strip().lower()
    if lang == "mandarin":
        return "chinese"
    return lang

def make_reference_set(x):
    if pd.isna(x):
        return set()

    langs = {
        str(lang).strip().lower()
        for lang in str(x).split(";")
        if str(lang).strip()
    }

    langs -= {"neutral", "mixed", "unclear"}

    return {
        normalise_language(lang)
        for lang in langs
    }

def parse_top5(x):
    if pd.isna(x):
        return []

    if isinstance(x, list):
        vals = x
    else:
        x = str(x).strip()

        try:
            vals = ast.literal_eval(x)
            if not isinstance(vals, list):
                vals = [vals]
        except:
            vals = [v.strip() for v in x.split(",")]

    return [
        normalise_language(v)
        for v in vals
        if str(v).strip()
    ]

df["split_clean"] = (
    df["split"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["speech_true"] = (
    df["speech_present"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["true_language_set"] = (
    df["langs_present"]
    .apply(make_reference_set)
)

df["n_valid_languages"] = (
    df["true_language_set"]
    .map(len)
)

df["top1"] = (
    df["whisper_med_prediction"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["top5"] = (
    df["top5_languages"]
    .apply(parse_top5)
)

multilingual = df[
    df["speech_true"].eq("yes")
    & df["n_valid_languages"].ge(2)
].copy()

multilingual["top1_any"] = multilingual.apply(
    lambda r: r["top1"] in r["true_language_set"],
    axis=1
)

multilingual["top5_any"] = multilingual.apply(
    lambda r: bool(
        set(r["top5"]) & r["true_language_set"]
    ),
    axis=1
)

multilingual["top5_all"] = multilingual.apply(
    lambda r: r["true_language_set"].issubset(
        set(r["top5"])
    ),
    axis=1
)

results = (
    multilingual
    .groupby("split_clean")
    .agg(
        n=("file_id", "size"),
        top1_any=("top1_any", "mean"),
        top5_any=("top5_any", "mean"),
        top5_all=("top5_all", "mean")
    )
    .reset_index()
)

for col in [
    "top1_any",
    "top5_any",
    "top5_all"
]:
    results[col] *= 100

print(
    results.to_string(
        index=False,
        formatters={
            "top1_any": "{:.2f}".format,
            "top5_any": "{:.2f}".format,
            "top5_all": "{:.2f}".format
        }
    )
)


In [ ]:
df = pd.read_csv("whisper_medium_predictions.csv")

df["true_language"] = (
    df["langs_present"]
    .astype("string")
    .str.strip()
    .str.lower()
)

clean = df[
    df["split"].astype("string").str.strip().str.lower().eq("dev")
    & df["speech_present"].astype("string").str.strip().str.lower().eq("yes")
    & df["speech_type"].astype("string").str.strip().str.lower().eq("live")
    & df["true_language"].notna()
    & df["true_language"].ne("")
    & ~df["true_language"].str.contains(";", na=False)
    & ~df["true_language"].isin(
        {"neutral", "unclear", "mixed", "cantonese"}
    )
].copy()

clean["pred_language"] = (
    clean["whisper_med_prediction"]
    .astype("string")
    .str.strip()
    .str.lower()
)

clean["target_language"] = (
    clean["true_language"]
    .replace({"mandarin": "chinese"})
)

clean["correct"] = (
    clean["pred_language"] == clean["target_language"]
)

family_col = (
    "family_id_clean"
    if "family_id_clean" in clean.columns
    else "family_id"
)

family_results = (
    clean.groupby(family_col, as_index=False)
    .agg(
        n=("file_id", "size"),
        correct=("correct", "sum"),
        accuracy=("correct", "mean")
    )
)

family_results["accuracy_pct"] = (
    family_results["accuracy"] * 100
)

family_results = family_results.sort_values(
    "accuracy_pct",
    ascending=False
).reset_index(drop=True)

pooled_accuracy = clean["correct"].mean() * 100
mean_family_accuracy = family_results["accuracy_pct"].mean()
sd_family_accuracy = family_results["accuracy_pct"].std()
min_family_accuracy = family_results["accuracy_pct"].min()
max_family_accuracy = family_results["accuracy_pct"].max()

print(
    family_results[
        [family_col, "n", "correct", "accuracy_pct"]
    ].to_string(
        index=False,
        formatters={
            "accuracy_pct": "{:.1f}".format
        }
    )
)

print(f"\nPooled accuracy: {pooled_accuracy:.1f}%")
print(f"Mean family accuracy: {mean_family_accuracy:.1f}%")
print(f"SD across families: {sd_family_accuracy:.1f} pp")
print(
    f"Family accuracy range: "
    f"{min_family_accuracy:.1f}%–{max_family_accuracy:.1f}%"
)


In [ ]:
family_language = (
    clean.groupby(
        ["true_language", family_col],
        as_index=False
    )
    .agg(
        n=("file_id", "size"),
        correct=("correct", "sum"),
        accuracy=("correct", "mean")
    )
)

family_language["accuracy_pct"] = (
    family_language["accuracy"] * 100
)

eligible = family_language[
    family_language["n"] >= 10
].copy()

language_summary = (
    eligible.groupby("true_language")
    .agg(
        n_families=(family_col, "nunique"),
        total_n=("n", "sum"),
        min_family_accuracy=("accuracy_pct", "min"),
        max_family_accuracy=("accuracy_pct", "max")
    )
)

language_summary["range_pp"] = (
    language_summary["max_family_accuracy"]
    - language_summary["min_family_accuracy"]
)

language_summary = language_summary[
    language_summary["n_families"] >= 2
].sort_values(
    "range_pp",
    ascending=False
)

print("\nLANGUAGES ELIGIBLE FOR WITHIN-LANGUAGE FAMILY COMPARISON")
print(language_summary.round(1))

print("\nFAMILY-LEVEL RESULTS")
print(
    eligible[
        eligible["true_language"].isin(language_summary.index)
    ]
    .sort_values(
        ["true_language", "accuracy_pct"],
        ascending=[True, False]
    )
    [[
        "true_language",
        family_col,
        "n",
        "correct",
        "accuracy_pct"
    ]]
    .to_string(
        index=False,
        formatters={"accuracy_pct": "{:.1f}".format}
    )
)


In [ ]:
test_results = []

for language in language_summary.index:
    lang = eligible[
        eligible["true_language"] == language
    ].copy()

    table = pd.DataFrame({
        "correct": lang["correct"].astype(int).to_numpy(),
        "incorrect": (lang["n"] - lang["correct"]).astype(int).to_numpy()
    })

    if len(lang) == 2:
        _, p = fisher_exact(table.to_numpy())
        test = "Fisher exact"
    else:
        _, p, _, _ = chi2_contingency(table.to_numpy())
        test = "Chi-square"

    test_results.append({
        "language": language,
        "n_families": len(lang),
        "total_n": lang["n"].sum(),
        "test": test,
        "p_raw": p,
        "min_accuracy": lang["accuracy_pct"].min(),
        "max_accuracy": lang["accuracy_pct"].max(),
        "range_pp": (
            lang["accuracy_pct"].max()
            - lang["accuracy_pct"].min()
        )
    })

family_tests = pd.DataFrame(test_results)

family_tests["p_holm"] = multipletests(
    family_tests["p_raw"],
    method="holm"
)[1]

family_tests["significant"] = (
    family_tests["p_holm"] < 0.05
)

family_tests["language"] = (
    family_tests["language"]
    .str.title()
)

family_tests = family_tests.sort_values(
    "p_holm"
).reset_index(drop=True)

print(
    family_tests.to_string(
        index=False,
        formatters={
            "p_raw": "{:.4f}".format,
            "p_holm": "{:.4f}".format,
            "min_accuracy": "{:.1f}".format,
            "max_accuracy": "{:.1f}".format,
            "range_pp": "{:.1f}".format,
        }
    )
)
